In [5]:
# STEP 1: Load the external knowledgebase and convert into pages.
# Example - Knowledgebase in pdf format
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("Python_Programming.pdf")
pdf_pages = loader.load()
print(f'No of pages: {len(pdf_pages)}')

No of pages: 140


In [6]:
# STEP 2: Split pages into chunks
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Spilit pages into small chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ",", " "]
)
split_docs = splitter.split_documents(pdf_pages)
print(f'Total no of chunks {len(split_docs)}')

Total no of chunks 528


In [7]:
# STEP 3: Create vector embeddings using FAISS

from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

embedding_model = OllamaEmbeddings(model="nomic-embed-text")
vector_store = FAISS.from_documents(split_docs, embedding=embedding_model)
vector_store.save_local("faiss_python_book")
print('python book vector store is created using FAISS...')



python book vector store is created using FAISS...


In [ ]:
# STEP 3: Create vector embeddings using Chroma
from langchain_community.vectorstores import Chroma

persist_directory = "chroma_python_book"

vector_store = Chroma.from_documents(
    documents=split_docs,
    embedding=embedding_model,
    persist_directory=persist_directory
)

print('python book vector store is created using Chroma...')

In [ ]:
# STEP 4: Build Retrieval Chain

from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import RetrievalQA
from langchain_ollama import ChatOllama

model = ChatOllama(model="gpt-oss:120b-cloud")
print("Model is ready")

USER_PROMPT_TEMPLATE = """Use the following pieces of the context to answer user's question.
If you don't know the answer, just say you don't know, don't try to make up the answer.
------------------------
{context}
Question: {question}
"""

USER_PROMPT = PromptTemplate(
    template=USER_PROMPT_TEMPLATE,
    input_variables=["context", "question"]
)
qa_chain = RetrievalQA.from_chain_type(
    model,
    chain_type="stuff",
    retriever=vector_store.as_retriever(),
    return_source_documents=True,
    chain_type_kwargs={"prompt": USER_PROMPT}
)
print(qa_chain)
print('Retrieval chain is built successfully...')

Model is ready
verbose=False combine_documents_chain=StuffDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="Use the following pieces of the context to answer user's question.\nIf you don't know the answer, just say you don't know, don't try to make up the answer.\n------------------------\n{context}\nQuestion: {question}\n"), llm=ChatOllama(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.0'}}, output_version=None, model='gpt-oss:120b-cloud'), output_parser=StrOutputParser(), llm_kwargs={}), document_prompt=PromptTemplate(input_variables=['page_content'], input_types={}, partial_variables={}, template='{page_content}'), document_variable_name='context') return_source_documents=True retriever=VectorStoreRetriever(tags=['FAISS', 'OllamaEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000017DE2707050>, search

In [12]:
# STEP 5: Build UI interface

import gradio as gr

def chatbot_response(message, history):
    # Use the created RetrievalQA chain to get the answer
    response = qa_chain({"query": message})
    answer = response["result"]
    source_documents = response["source_documents"]

    # Format the response to include the answer and source documents (optional)
    formatted_response = f"{answer}" # You can add source documents here if desired

    print(f"Source documents: {source_documents}")

    return formatted_response

# Create the Gradio interface
iface = gr.ChatInterface(
    fn=chatbot_response,
    title="RAG Python Chatbot",
    description="Ask questions about Python programming."
)

# Launch the interface
iface.launch(share=True, server_port=7861)

* Running on local URL:  http://127.0.0.1:7861

Could not create share link. Missing file: C:\Users\ak60492\.cache\huggingface\gradio\frpc\frpc_windows_amd64_v0.3. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.3/frpc_windows_amd64.exe
2. Rename the downloaded file to: frpc_windows_amd64_v0.3
3. Move the file to this location: C:\Users\ak60492\.cache\huggingface\gradio\frpc


c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Source documents: [Document(id='6d82a7d4-1cab-4761-b6e7-dbbf309fe5de', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'trapped': '/False', 'source': 'Python_Programming.pdf', 'total_pages': 140, 'page': 19, 'page_label': '20'}, page_content='Chapter 2\nWhat is Python?\n2.1 Introduction to Python\nPython is an open source and cross-platform programming language, that has\nbecome increasingly popular over the last ten years. It was ﬁrst released in\n1991. Latest version is 3.7.0. CPython is the reference implementation of the'), Document(id='718e16d4-15f5-4fe9-8bf8-5570f5110b45', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Source documents: [Document(id='6a09d191-ecfd-452d-99d2-d4b65d97b78c', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'trapped': '/False', 'source': 'Python_Programming.pdf', 'total_pages': 140, 'page': 29, 'page_label': '30'}, page_content='Below we see how we can run Python from the Console which is part of the OS.\n27'), Document(id='9be66340-225d-484b-9efd-4fe5f059215b', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'trapped': '/False', 'source': 'Python_Programming.pdf', 'total_pages': 140, 'page': 8, 'page_label': '9'}, page_content='3.4 Running Python from the Console . . . . .

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Source documents: [Document(id='669a9367-d459-42f2-8c86-354fcca7da03', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'trapped': '/False', 'source': 'Python_Programming.pdf', 'total_pages': 140, 'page': 54, 'page_label': '55'}, page_content='Listing 5.3: Using Arrays in Python\nNote! Python uses ”elif” not ”elseif” like many other programming languages\ndo.\n[End of Example]\n5.2 Arrays\nAn array is a special variable, which can hold more than one value at a time.\nHere are some Examples how you can create and use Arrays in Python:'), Document(id='d8381811-392d-4d82-9654-d8df2c9fed98', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Source documents: [Document(id='8761c716-59b5-469c-ad05-03cc4daac71f', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'trapped': '/False', 'source': 'Python_Programming.pdf', 'total_pages': 140, 'page': 32, 'page_label': '33'}, page_content='The Default Location is:\nC:\\ U s e r s\\ u s e r\\AppData\\ L o c a l\\ Programs \\ Python\\ Python37 −32\\\nClick Save and open the Command Prompt once more and enter ”python” to\nverify it works. See Figure 3.3.\n30'), Document(id='f0e669ac-9ea7-491a-8c9d-8336e63ee4c6', metadata={'producer': 'pdfTeX-1.40.18', 'creator': 'TeX', 'creationdate': '2020-08-12T08:23:16+00:00', 'moddate': '2020-08-12T08:23:16+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.14159265-2.6-1.40.18 (TeX Live 2017) kpathsea version 6.2.3', 'trapped': '/False', 'source

c:\Users\ak60492\AppData\Local\Programs\Python\Python312\Lib\site-packages\gradio\routes.py:1541: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
